In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git lfs version || (apt-get update -qq && apt-get install -y -qq git-lfs)

git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	log
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package git-lfs.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [18]:

from google.colab import userdata
from pathlib import Path
import subprocess
import tempfile
import uuid
import os

# ============================================================
# 1. Configure GitHub repository and user information
# ============================================================

REPO_URL = "https://github.com/ACKarl/TTS.git"

GIT_NAME = "ACKarl"
GIT_EMAIL = "104746285+ACKarl@users.noreply.github.com"

# Load the GitHub access token from Colab Secrets.
token = userdata.get("GITHUB_TOKEN")

assert token, "GitHub token is missing."

token = token.strip()

# ============================================================
# 2. Create a temporary working directory
# ============================================================

with tempfile.TemporaryDirectory(dir="/content") as temp:

    temp_path = Path(temp)

    # ========================================================
    # 3. Configure secure Git authentication
    # ========================================================

    # Create a temporary authentication script.
    askpass = temp_path / "askpass.sh"

    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "x-access-token\\n" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )

    askpass.chmod(0o700)

    # Configure environment variables.
    env = os.environ.copy()

    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    # Avoid downloading existing LFS objects.
    env["GIT_LFS_SKIP_SMUDGE"] = "1"

    # ========================================================
    # 4. Define a Git execution helper
    # ========================================================

    def run_git(*args, cwd=None, check=True):

        # Disable cached credentials and authentication headers.
        command = [
            "git",
            "-c", "credential.helper=",
            "-c", "http.https://github.com/.extraheader=",
            *args
        ]

        # Execute the Git command.
        result = subprocess.run(
            command,
            cwd=cwd,
            env=env,
            capture_output=True,
            text=True,
            errors="replace"
        )

        # Remove the token from any displayed output.
        def redact(value):
            return value.replace(token, "[REDACTED]")

        # Display errors if the Git command fails.
        if result.returncode != 0:

            print("\nGit operation failed.")
            print("Command:", "git", " ".join(args))

            if result.stdout:
                print("\nSTDOUT:")
                print(redact(result.stdout))

            if result.stderr:
                print("\nSTDERR:")
                print(redact(result.stderr))

            if check:
                raise RuntimeError(
                    f"Git exited with code {result.returncode}"
                )

        return result

    # ========================================================
    # 5. Clone the GitHub repository
    # ========================================================

    print("Step 1: Cloning repository...")

    repo_dir = temp_path / "repository"

    run_git(
        "clone",
        "--depth", "1",
        REPO_URL,
        str(repo_dir)
    )

    print("Repository cloned successfully.")

    # ========================================================
    # 6. Configure Git user identity
    # ========================================================

    run_git(
        "config",
        "user.name",
        GIT_NAME,
        cwd=repo_dir
    )

    run_git(
        "config",
        "user.email",
        GIT_EMAIL,
        cwd=repo_dir
    )

    # ========================================================
    # 7. Create a temporary test branch
    # ========================================================

    print("\nStep 2: Creating temporary test branch...")

    branch_name = (
        "colab-push-test-" + uuid.uuid4().hex[:8]
    )

    run_git(
        "checkout",
        "-b",
        branch_name,
        cwd=repo_dir
    )

    print("Test branch:", branch_name)

    # ========================================================
    # 8. Create a small test file
    # ========================================================

    print("\nStep 3: Creating test file...")

    test_file = repo_dir / "colab_push_test.txt"

    test_file.write_text(
        "This file is used to test GitHub Push from Colab.\n",
        encoding="utf-8"
    )

    # Stage the test file.
    run_git(
        "add",
        "colab_push_test.txt",
        cwd=repo_dir
    )

    # Commit the test file.
    run_git(
        "commit",
        "-m",
        "Test GitHub Push from Colab",
        cwd=repo_dir
    )

    print("Test commit created successfully.")

    # ========================================================
    # 9. Test Git Push
    # ========================================================

    print("\nStep 4: Testing Git Push...")

    push_result = run_git(
        "push",
        "--progress",
        "origin",
        f"HEAD:refs/heads/{branch_name}",
        cwd=repo_dir,
        check=False
    )

    # ========================================================
    # 10. Evaluate the Push result
    # ========================================================

    if push_result.returncode == 0:

        print("\nSUCCESS: Small Git push works!")
        print("GitHub authentication is working.")

        # ====================================================
        # 11. Remove the temporary remote branch
        # ====================================================

        print("\nStep 5: Removing temporary test branch...")

        delete_result = run_git(
            "push",
            "origin",
            f":refs/heads/{branch_name}",
            cwd=repo_dir,
            check=False
        )

        if delete_result.returncode == 0:
            print("Test branch removed successfully.")
        else:
            print(
                "Push succeeded, but automatic branch "
                "deletion failed."
            )
            print("Branch to remove:", branch_name)

    else:

        print("\nFAILED: Git Push was rejected.")
        print("Git exit code:", push_result.returncode)

        print(
            "\nReview the Git error above to identify "
            "the exact cause."
        )

    # ========================================================
    # 12. Display final status
    # ========================================================

    print("\n" + "=" * 50)
    print("GitHub Push Test Finished")
    print("Repository:", REPO_URL)

    if push_result.returncode == 0:
        print("Push status: SUCCESS")
    else:
        print("Push status: FAILED")

    print("=" * 50)


Step 1: Cloning repository...
Repository cloned successfully.

Step 2: Creating temporary test branch...
Test branch: colab-push-test-9a5fd0ea

Step 3: Creating test file...
Test commit created successfully.

Step 4: Testing Git Push...

SUCCESS: Small Git push works!
GitHub authentication is working.

Step 5: Removing temporary test branch...

Git operation failed.
Command: git push origin :refs/heads/colab-push-test-9a5fd0ea

STDERR:
To https://github.com/ACKarl/TTS.git
 ! [remote rejected] colab-push-test-9a5fd0ea (refusing to delete the current branch: refs/heads/colab-push-test-9a5fd0ea)
error: failed to push some refs to 'https://github.com/ACKarl/TTS.git'

Push succeeded, but automatic branch deletion failed.
Branch to remove: colab-push-test-9a5fd0ea

GitHub Push Test Finished
Repository: https://github.com/ACKarl/TTS.git
Push status: SUCCESS
